# Traceability — manuscript ↔ artifacts ↔ pipeline

Every number in the manuscript's exhibits is regenerated or verified from the aggregate artifacts in `artifacts/`; no licensed microdata is used or required (see `DATA_ACCESS.md`).
Outputs are stored, so the notebooks render on GitHub without running. Rebuild with `python3 code/make_notebooks.py`.


## Setup


In [ ]:
import sys, runpy
sys.path.insert(0, "code"); import exhibits_lib as X
md = open("manuscript/PE_Labour_Adjustment_manuscript_2026-08-31.md", encoding="utf-8").read()
paper, appendix = X.split_document(md)
pool = X.load_pool("artifacts")
print("artifact pool loaded ·", len(pool), "quantized values")


artifact pool loaded · 56854 quantized values


## Every decimal token in the manuscript traces to the artifacts
References and six-figure display roundings aside, every decimal in the prose and tables must exist in the artifact pool at its displayed precision.


In [ ]:
import re
body = re.split(r"\n#+ References\s*\n", md)[0]
tot = bad = 0; badl = []
for m in X.NUM.finditer(body.replace("−", "-").replace("+", "")):
    x = float(m.group()); nd = len(m.group().split(".")[1]); tot += 1
    if (nd, X._q(x, nd)) not in pool and (nd, X._q(abs(x), nd)) not in pool:
        bad += 1; badl.append(m.group())
print(f"decimal tokens {tot} · unmatched {bad}"); print(badl[:8])
assert bad <= 1, badl   # DOI in the reference heading area at most


decimal tokens 1199 · unmatched 0
[]


## The claims ledger resolves against the artifacts


In [ ]:
import csv, json, os
rows = list(csv.DictReader(open("artifacts/CLAIMS_LEDGER.csv", encoding="utf-8-sig")))
def resolve(o, p):
    for k in [k for k in p.split(".") if k]: o = o[int(k)] if isinstance(o, list) else o[k]
    return o
exact = derived = mismatch = missing = 0
for r in rows:
    f = os.path.join("artifacts", os.path.basename(r["source_json"]))
    if not os.path.exists(f): missing += 1; continue
    try: o = resolve(json.load(open(f, encoding="utf-8")), r["json_path"])
    except Exception: mismatch += 1; continue
    if isinstance(o, (dict, list)): derived += 1; continue
    try: ok = abs(float(r["value"]) - float(o)) <= max(5e-5, abs(float(o)) * 1e-6)
    except Exception: ok = str(o) == r["value"]
    exact += ok; mismatch += (not ok)
print(f"claims ledger: {len(rows)} rows · exact {exact} · derived {derived} · mismatch {mismatch} · missing {missing}")
assert mismatch == 0


claims ledger: 370 rows · exact 364 · derived 6 · mismatch 0 · missing 0
